In [8]:
from pathlib import Path
import sys
import gc
import time

import numpy as np
import pandas as pd

# Locate project root: the folder containing common.py
cwd = Path.cwd()

if (cwd / "common.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "common.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "common.py not found. Run this notebook from "
        "the project root or its baseline/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import (
    LANGS,
    RUN_DIR,
    RES_DIR,
    load_benchmark,
    record,
    rrf_fuse,
)

print("Project root:", PROJECT_ROOT)
print("Runs directory:", RUN_DIR)
print("Results directory:", RES_DIR)
print("Languages:", LANGS)
print("Number of languages:", len(LANGS))

assert len(LANGS) == 13
assert RUN_DIR.exists()

Project root: c:\Users\sanka\OneDrive\Desktop\research paper
Runs directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs
Results directory: C:\Users\sanka\OneDrive\Desktop\research paper\results
Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Number of languages: 13


In [9]:
print("Available run directories:\n")

for folder in sorted(RUN_DIR.iterdir()):
    if not folder.is_dir():
        continue

    score_files = list(folder.glob("*_scores.npy"))
    rank_files = list(folder.glob("*_ranks.npy"))

    print(
        f"{folder.name:25s} "
        f"scores={len(score_files):2d} "
        f"ranks={len(rank_files):2d}"
    )

Available run directories:

bge-m3                    scores=13 ranks=13
bm25                      scores=13 ranks=13
indicbert-v3              scores= 1 ranks= 1
mE5-base                  scores=13 ranks=13
mE5-large                 scores=13 ranks=13
mE5-small                 scores=13 ranks=13
muril                     scores=13 ranks=13


In [10]:
# Change this only if your BM25 directory has another name.
BM25_TAG = "bm25"

BGE_TAG = "bge-m3"

# RRF hyperparameter
RRF_K = 60

# Fuse rankings using all documents in the corpus.
RRF_DEPTH = None

# Model tag used for the hybrid outputs.
HYBRID_TAG = "bm25+bge-m3-rrf"

BM25_DIR = RUN_DIR / BM25_TAG
BGE_DIR = RUN_DIR / BGE_TAG
HYBRID_DIR = RUN_DIR / HYBRID_TAG

HYBRID_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print("BM25 directory:", BM25_DIR)
print("BGE directory:", BGE_DIR)
print("Hybrid output directory:", HYBRID_DIR)
print("RRF k:", RRF_K)
print("RRF depth:", RRF_DEPTH)

assert BM25_DIR.is_dir(), f"BM25 directory not found: {BM25_DIR}"
assert BGE_DIR.is_dir(), f"BGE directory not found: {BGE_DIR}"

BM25 directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25
BGE directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bge-m3
Hybrid output directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25+bge-m3-rrf
RRF k: 60
RRF depth: None


In [11]:
validation_errors = []

for lang in LANGS:
    bm25_scores_path = BM25_DIR / f"{lang}_scores.npy"
    bge_scores_path = BGE_DIR / f"{lang}_scores.npy"

    bm25_ranks_path = BM25_DIR / f"{lang}_ranks.npy"
    bge_ranks_path = BGE_DIR / f"{lang}_ranks.npy"

    required_paths = [
        bm25_scores_path,
        bge_scores_path,
        bm25_ranks_path,
        bge_ranks_path,
    ]

    missing = [p.name for p in required_paths if not p.exists()]

    if missing:
        validation_errors.append(
            f"{lang}: missing files {missing}"
        )
        continue

    queries, corpus, gold = load_benchmark(lang)

    bm25_scores = np.load(bm25_scores_path, mmap_mode="r")
    bge_scores = np.load(bge_scores_path, mmap_mode="r")

    bm25_ranks = np.load(bm25_ranks_path, mmap_mode="r")
    bge_ranks = np.load(bge_ranks_path, mmap_mode="r")

    expected_scores_shape = (len(queries), len(corpus))
    expected_ranks_shape = (len(queries),)

    if bm25_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: BM25 shape {bm25_scores.shape}, "
            f"expected {expected_scores_shape}"
        )

    if bge_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: BGE shape {bge_scores.shape}, "
            f"expected {expected_scores_shape}"
        )

    if bm25_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: BM25 ranks shape {bm25_ranks.shape}"
        )

    if bge_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: BGE ranks shape {bge_ranks.shape}"
        )

    if bm25_scores.shape == bge_scores.shape == expected_scores_shape:
        print(
            f"{lang}: queries={len(queries)}, "
            f"documents={len(corpus)}, "
            f"shape={bm25_scores.shape}"
        )

    del bm25_scores, bge_scores, bm25_ranks, bge_ranks

if validation_errors:
    print("\nVALIDATION FAILED:")
    for error in validation_errors:
        print("-", error)

    raise RuntimeError(
        "Fix the input files before running hybrid retrieval."
    )

print("\nPASS: all 13 languages have compatible input files.")

as: queries=1000, documents=1000, shape=(1000, 1000)
bn: queries=1000, documents=1000, shape=(1000, 1000)
gu: queries=999, documents=999, shape=(999, 999)
hi: queries=1000, documents=1000, shape=(1000, 1000)
kn: queries=1000, documents=1000, shape=(1000, 1000)
ml: queries=1000, documents=1000, shape=(1000, 1000)
mr: queries=1000, documents=1000, shape=(1000, 1000)
ne: queries=1000, documents=1000, shape=(1000, 1000)
or: queries=1000, documents=1000, shape=(1000, 1000)
pa: queries=1000, documents=1000, shape=(1000, 1000)
ta: queries=1000, documents=1000, shape=(1000, 1000)
te: queries=1000, documents=1000, shape=(1000, 1000)
ur: queries=1000, documents=1000, shape=(1000, 1000)

PASS: all 13 languages have compatible input files.


In [12]:
lang = "hi"

bm25_scores = np.load(BM25_DIR / f"{lang}_scores.npy")
bge_scores = np.load(BGE_DIR / f"{lang}_scores.npy")

queries, corpus, gold = load_benchmark(lang)

assert bm25_scores.shape == bge_scores.shape
assert bm25_scores.shape == (len(queries), len(corpus))
assert len(gold) == len(queries)

t0 = time.time()

hybrid_scores = rrf_fuse(
    score_mats=[bm25_scores, bge_scores],
    k=RRF_K,
    depth=RRF_DEPTH,
)

elapsed = time.time() - t0

assert hybrid_scores.shape == bm25_scores.shape
assert np.isfinite(hybrid_scores).all()

# Verify the fused ranking can be recorded correctly.
hybrid_metrics = record(
    model=HYBRID_TAG,
    key=lang,
    S=hybrid_scores,
    gold=gold,
    setting="mono",
)

print(f"Language: {lang}")
print(f"Queries: {len(queries)}")
print(f"Documents: {len(corpus)}")
print(f"Hybrid matrix shape: {hybrid_scores.shape}")
print(f"Fusion time: {elapsed:.3f} seconds")
print(f"MRR: {hybrid_metrics['MRR']:.4f}")
print(f"MRR@10: {hybrid_metrics['MRR@10']:.4f}")
print(f"Recall@1: {hybrid_metrics['R@1']:.4f}")
print(f"Recall@5: {hybrid_metrics['R@5']:.4f}")
print(f"Recall@10: {hybrid_metrics['R@10']:.4f}")

Language: hi
Queries: 1000
Documents: 1000
Hybrid matrix shape: (1000, 1000)
Fusion time: 0.252 seconds
MRR: 0.8005
MRR@10: 0.7967
Recall@1: 0.7310
Recall@5: 0.8770
Recall@10: 0.9060


In [13]:
all_metrics = []
failed_languages = []

for lang in LANGS:
    print("\n" + "=" * 65)
    print(f"Running BM25 + BGE-M3 RRF: {lang}")
    print("=" * 65)

    try:
        start = time.time()

        # Load saved baseline score matrices.
        bm25_scores = np.load(
            BM25_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        bge_scores = np.load(
            BGE_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        # Validate against the canonical benchmark.
        queries, corpus, gold = load_benchmark(lang)

        expected_shape = (len(queries), len(corpus))

        assert bm25_scores.shape == expected_shape
        assert bge_scores.shape == expected_shape
        assert len(gold) == len(queries)

        # Reciprocal Rank Fusion.
        hybrid_scores = rrf_fuse(
            score_mats=[bm25_scores, bge_scores],
            k=RRF_K,
            depth=RRF_DEPTH,
        )

        assert hybrid_scores.shape == expected_shape
        assert np.isfinite(hybrid_scores).all()

        # Save fused scores, ranks and metrics.
        metrics = record(
            model=HYBRID_TAG,
            key=lang,
            S=hybrid_scores,
            gold=gold,
            setting="mono",
        )

        all_metrics.append(metrics)

        print(f"MRR:       {metrics['MRR']:.4f}")
        print(f"MRR@10:    {metrics['MRR@10']:.4f}")
        print(f"Recall@1:  {metrics['R@1']:.4f}")
        print(f"Recall@5:  {metrics['R@5']:.4f}")
        print(f"Recall@10: {metrics['R@10']:.4f}")
        print(f"Recall@20: {metrics['R@20']:.4f}")
        print(f"Recall@100:{metrics['R@100']:.4f}")
        print(f"Time: {(time.time() - start):.2f} seconds")

        del bm25_scores, bge_scores, hybrid_scores
        gc.collect()

    except Exception as exc:
        print(f"FAILED: {lang}")
        print(f"Error: {exc}")
        failed_languages.append(lang)
        gc.collect()

print("\n" + "=" * 65)
print("HYBRID RRF RUN FINISHED")
print("=" * 65)
print("Successful this session:", len(all_metrics))
print("Failed languages:", failed_languages or "None")


Running BM25 + BGE-M3 RRF: as
MRR:       0.6094
MRR@10:    0.5975
Recall@1:  0.5290
Recall@5:  0.6830
Recall@10: 0.7440
Recall@20: 0.8540
Recall@100:0.9710
Time: 0.33 seconds

Running BM25 + BGE-M3 RRF: bn
MRR:       0.7397
MRR@10:    0.7337
Recall@1:  0.6650
Recall@5:  0.8240
Recall@10: 0.8590
Recall@20: 0.9130
Recall@100:0.9820
Time: 0.27 seconds

Running BM25 + BGE-M3 RRF: gu
MRR:       0.6752
MRR@10:    0.6671
Recall@1:  0.5926
Recall@5:  0.7588
Recall@10: 0.8128
Recall@20: 0.8919
Recall@100:0.9780
Time: 0.28 seconds

Running BM25 + BGE-M3 RRF: hi
MRR:       0.8005
MRR@10:    0.7967
Recall@1:  0.7310
Recall@5:  0.8770
Recall@10: 0.9060
Recall@20: 0.9410
Recall@100:0.9860
Time: 0.30 seconds

Running BM25 + BGE-M3 RRF: kn
MRR:       0.6542
MRR@10:    0.6426
Recall@1:  0.5860
Recall@5:  0.7090
Recall@10: 0.7620
Recall@20: 0.8890
Recall@100:0.9710
Time: 0.24 seconds

Running BM25 + BGE-M3 RRF: ml
MRR:       0.6515
MRR@10:    0.6418
Recall@1:  0.5770
Recall@5:  0.7230
Recall@10: 0.7850

In [14]:
verification_errors = []

for lang in LANGS:
    scores_path = HYBRID_DIR / f"{lang}_scores.npy"
    ranks_path = HYBRID_DIR / f"{lang}_ranks.npy"

    if not scores_path.exists():
        verification_errors.append(f"{lang}: missing scores file")
        continue

    if not ranks_path.exists():
        verification_errors.append(f"{lang}: missing ranks file")
        continue

    queries, corpus, gold = load_benchmark(lang)

    scores = np.load(scores_path, mmap_mode="r")
    ranks = np.load(ranks_path)

    if scores.shape != (len(queries), len(corpus)):
        verification_errors.append(
            f"{lang}: incorrect score shape {scores.shape}"
        )

    if ranks.shape != (len(queries),):
        verification_errors.append(
            f"{lang}: incorrect rank shape {ranks.shape}"
        )

    if not np.isfinite(scores).all():
        verification_errors.append(
            f"{lang}: non-finite scores"
        )

    if np.any(ranks < 1) or np.any(ranks > len(corpus)):
        verification_errors.append(
            f"{lang}: invalid gold-document ranks"
        )

    print(
        f"{lang}: scores={scores.shape}, "
        f"ranks={ranks.shape}, verified"
    )

print("\nScore files:",
      sum((HYBRID_DIR / f"{l}_scores.npy").exists() for l in LANGS),
      "/ 13")

print("Rank files:",
      sum((HYBRID_DIR / f"{l}_ranks.npy").exists() for l in LANGS),
      "/ 13")

if verification_errors:
    for error in verification_errors:
        print("ERROR:", error)
    raise RuntimeError("Output verification failed.")

print("\nPASS: all hybrid output files passed verification.")

as: scores=(1000, 1000), ranks=(1000,), verified
bn: scores=(1000, 1000), ranks=(1000,), verified
gu: scores=(999, 999), ranks=(999,), verified
hi: scores=(1000, 1000), ranks=(1000,), verified
kn: scores=(1000, 1000), ranks=(1000,), verified
ml: scores=(1000, 1000), ranks=(1000,), verified
mr: scores=(1000, 1000), ranks=(1000,), verified
ne: scores=(1000, 1000), ranks=(1000,), verified
or: scores=(1000, 1000), ranks=(1000,), verified
pa: scores=(1000, 1000), ranks=(1000,), verified
ta: scores=(1000, 1000), ranks=(1000,), verified
te: scores=(1000, 1000), ranks=(1000,), verified
ur: scores=(1000, 1000), ranks=(1000,), verified

Score files: 13 / 13
Rank files: 13 / 13

PASS: all hybrid output files passed verification.


In [15]:
metrics_path = RES_DIR / "metrics.csv"

assert metrics_path.exists(), "metrics.csv not found."

metrics_df = pd.read_csv(metrics_path)

required_models = [BM25_TAG, BGE_TAG, HYBRID_TAG]

comparison = metrics_df[
    (metrics_df["model"].isin(required_models))
    & (metrics_df["setting"] == "mono")
].copy()

metric_columns = [
    "MRR",
    "MRR@10",
    "R@1",
    "R@5",
    "R@10",
    "R@20",
    "R@100",
]

comparison = comparison[
    comparison["key"].isin(LANGS)
]

# Keep one row per model and language.
comparison = comparison.drop_duplicates(
    subset=["model", "key", "setting"],
    keep="last",
)

for model_name in required_models:
    present = set(
        comparison.loc[
            comparison["model"] == model_name, "key"
        ]
    )
    missing = set(LANGS) - present

    print(f"{model_name}: {len(present)}/13 languages")
    if missing:
        print("  Missing:", sorted(missing))

# Pivot MRR for direct per-language comparison.
mrr_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="MRR",
).reindex(LANGS)

print("\nMRR comparison:")
display(mrr_comparison.round(4))

# Pivot Recall@10.
r10_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="R@10",
).reindex(LANGS)

print("\nRecall@10 comparison:")
display(r10_comparison.round(4))

bm25: 13/13 languages
bge-m3: 13/13 languages
bm25+bge-m3-rrf: 13/13 languages

MRR comparison:


model,bge-m3,bm25,bm25+bge-m3-rrf
key,,,
as,0.8073,0.4616,0.6094
bn,0.8602,0.6189,0.7397
gu,0.8336,0.5465,0.6752
hi,0.8863,0.7117,0.8005
kn,0.8379,0.5380,0.6542
ml,0.8405,0.5248,0.6515
mr,0.8521,0.5097,0.6683
ne,0.8504,0.5052,0.6557
or,0.7953,0.5172,0.6477



Recall@10 comparison:


model,bge-m3,bm25,bm25+bge-m3-rrf
key,,,
as,0.9110,0.5990,0.7440
bn,0.9430,0.7720,0.8590
gu,0.9369,0.6877,0.8128
hi,0.9570,0.8420,0.9060
kn,0.9200,0.6630,0.7620
ml,0.9270,0.6620,0.7850
mr,0.9390,0.6630,0.7850
ne,0.9440,0.6610,0.8090
or,0.9050,0.6520,0.7830


In [16]:
if not all(
    model_name in mrr_comparison.columns
    for model_name in required_models
):
    raise RuntimeError(
        "Cannot compare all three systems. "
        "Check the missing model rows in Cell 8."
    )

delta = pd.DataFrame(index=LANGS)

delta["BM25_MRR"] = mrr_comparison[BM25_TAG]
delta["BGE_MRR"] = mrr_comparison[BGE_TAG]
delta["Hybrid_MRR"] = mrr_comparison[HYBRID_TAG]

delta["Hybrid_minus_BM25"] = (
    delta["Hybrid_MRR"] - delta["BM25_MRR"]
)

delta["Hybrid_minus_BGE"] = (
    delta["Hybrid_MRR"] - delta["BGE_MRR"]
)

delta["Best_baseline_MRR"] = delta[
    ["BM25_MRR", "BGE_MRR"]
].max(axis=1)

delta["Hybrid_beats_best_baseline"] = (
    delta["Hybrid_MRR"] > delta["Best_baseline_MRR"]
)

display(delta.round(4))

print(
    "Languages where hybrid beats both baselines:",
    int(delta["Hybrid_beats_best_baseline"].sum()),
    "/ 13",
)

print("\nMacro-average MRR:")
print(delta[
    ["BM25_MRR", "BGE_MRR", "Hybrid_MRR"]
].mean().round(4))

,BM25_MRR,BGE_MRR,Hybrid_MRR,Hybrid_minus_BM25,Hybrid_minus_BGE,Best_baseline_MRR,Hybrid_beats_best_baseline
as,0.4616,0.8073,0.6094,0.1477,-0.1979,0.8073,False
bn,0.6189,0.8602,0.7397,0.1208,-0.1205,0.8602,False
gu,0.5465,0.8336,0.6752,0.1286,-0.1584,0.8336,False
hi,0.7117,0.8863,0.8005,0.0888,-0.0858,0.8863,False
kn,0.5380,0.8379,0.6542,0.1162,-0.1837,0.8379,False
ml,0.5248,0.8405,0.6515,0.1266,-0.1890,0.8405,False
mr,0.5097,0.8521,0.6683,0.1586,-0.1838,0.8521,False
ne,0.5052,0.8504,0.6557,0.1505,-0.1948,0.8504,False
or,0.5172,0.7953,0.6477,0.1305,-0.1476,0.7953,False
pa,0.6532,0.8298,0.7530,0.0998,-0.0769,0.8298,False


Languages where hybrid beats both baselines: 0 / 13

Macro-average MRR:
BM25_MRR      0.5651
BGE_MRR       0.8384
Hybrid_MRR    0.6930
dtype: float64


In [31]:
from pathlib import Path
import sys
import gc
import time

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "common.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "common.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "common.py not found. Run this notebook from "
        "the project root or the baseline/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Import canonical project utilities
# ------------------------------------------------------------

from common import (
    LANGS,
    RUN_DIR,
    RES_DIR,
    load_benchmark,
    record,
    rrf_fuse,
)

print("Project root:", PROJECT_ROOT)
print("Runs directory:", RUN_DIR)
print("Results directory:", RES_DIR)
print("Languages:", LANGS)
print("Number of languages:", len(LANGS))

assert len(LANGS) == 13
assert RUN_DIR.exists()

Project root: c:\Users\sanka\OneDrive\Desktop\research paper
Runs directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs
Results directory: C:\Users\sanka\OneDrive\Desktop\research paper\results
Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Number of languages: 13


In [32]:
# ------------------------------------------------------------
# Input model directories
# ------------------------------------------------------------

BM25_TAG = "bm25"

E5_TAG = "mE5-base"

# ------------------------------------------------------------
# RRF configuration
# ------------------------------------------------------------

RRF_K = 60

# Use every document in the corpus.
RRF_DEPTH = None

# ------------------------------------------------------------
# Output tag
# ------------------------------------------------------------

HYBRID_TAG = "bm25+mE5-base-rrf"

BM25_DIR = RUN_DIR / BM25_TAG
E5_DIR = RUN_DIR / E5_TAG
HYBRID_DIR = RUN_DIR / HYBRID_TAG

HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("BM25 directory:", BM25_DIR)
print("E5 directory:", E5_DIR)
print("Hybrid output directory:", HYBRID_DIR)
print("RRF k:", RRF_K)
print("RRF depth:", RRF_DEPTH)

assert BM25_DIR.is_dir(), (
    f"BM25 directory not found: {BM25_DIR}"
)

assert E5_DIR.is_dir(), (
    f"E5 directory not found: {E5_DIR}"
)

BM25 directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25
E5 directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\mE5-base
Hybrid output directory: C:\Users\sanka\OneDrive\Desktop\research paper\runs\bm25+mE5-base-rrf
RRF k: 60
RRF depth: None


In [33]:
print("Available BM25 + E5 files:\n")

validation_errors = []

for lang in LANGS:

    bm25_scores_path = (
        BM25_DIR / f"{lang}_scores.npy"
    )

    e5_scores_path = (
        E5_DIR / f"{lang}_scores.npy"
    )

    bm25_ranks_path = (
        BM25_DIR / f"{lang}_ranks.npy"
    )

    e5_ranks_path = (
        E5_DIR / f"{lang}_ranks.npy"
    )

    required = [
        bm25_scores_path,
        e5_scores_path,
        bm25_ranks_path,
        e5_ranks_path,
    ]

    missing = [
        p.name
        for p in required
        if not p.exists()
    ]

    if missing:
        validation_errors.append(
            f"{lang}: missing {missing}"
        )
        continue

    print(
        f"{lang}: "
        f"BM25={bm25_scores_path.name} | "
        f"E5={e5_scores_path.name}"
    )

if validation_errors:

    print("\nVALIDATION FAILED:")

    for error in validation_errors:
        print("-", error)

    raise RuntimeError(
        "Fix the missing E5/BM25 files before continuing."
    )

print("\nPASS: all required files exist for all 13 languages.")

Available BM25 + E5 files:

as: BM25=as_scores.npy | E5=as_scores.npy
bn: BM25=bn_scores.npy | E5=bn_scores.npy
gu: BM25=gu_scores.npy | E5=gu_scores.npy
hi: BM25=hi_scores.npy | E5=hi_scores.npy
kn: BM25=kn_scores.npy | E5=kn_scores.npy
ml: BM25=ml_scores.npy | E5=ml_scores.npy
mr: BM25=mr_scores.npy | E5=mr_scores.npy
ne: BM25=ne_scores.npy | E5=ne_scores.npy
or: BM25=or_scores.npy | E5=or_scores.npy
pa: BM25=pa_scores.npy | E5=pa_scores.npy
ta: BM25=ta_scores.npy | E5=ta_scores.npy
te: BM25=te_scores.npy | E5=te_scores.npy
ur: BM25=ur_scores.npy | E5=ur_scores.npy

PASS: all required files exist for all 13 languages.


In [34]:
validation_errors = []

for lang in LANGS:

    queries, corpus, gold = load_benchmark(lang)

    expected_scores_shape = (
        len(queries),
        len(corpus),
    )

    expected_ranks_shape = (
        len(queries),
    )

    bm25_scores = np.load(
        BM25_DIR / f"{lang}_scores.npy",
        mmap_mode="r",
    )

    e5_scores = np.load(
        E5_DIR / f"{lang}_scores.npy",
        mmap_mode="r",
    )

    bm25_ranks = np.load(
        BM25_DIR / f"{lang}_ranks.npy",
        mmap_mode="r",
    )

    e5_ranks = np.load(
        E5_DIR / f"{lang}_ranks.npy",
        mmap_mode="r",
    )

    if bm25_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: BM25 shape "
            f"{bm25_scores.shape}, expected "
            f"{expected_scores_shape}"
        )

    if e5_scores.shape != expected_scores_shape:
        validation_errors.append(
            f"{lang}: E5 shape "
            f"{e5_scores.shape}, expected "
            f"{expected_scores_shape}"
        )

    if bm25_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: BM25 ranks shape "
            f"{bm25_ranks.shape}, expected "
            f"{expected_ranks_shape}"
        )

    if e5_ranks.shape != expected_ranks_shape:
        validation_errors.append(
            f"{lang}: E5 ranks shape "
            f"{e5_ranks.shape}, expected "
            f"{expected_ranks_shape}"
        )

    print(
        f"{lang}: "
        f"queries={len(queries)}, "
        f"documents={len(corpus)}, "
        f"BM25={bm25_scores.shape}, "
        f"E5={e5_scores.shape}"
    )

    del (
        bm25_scores,
        e5_scores,
        bm25_ranks,
        e5_ranks,
    )

if validation_errors:

    print("\nVALIDATION FAILED:")

    for error in validation_errors:
        print("-", error)

    raise RuntimeError(
        "Input validation failed."
    )

print(
    "\nPASS: BM25 and E5 matrices match "
    "the canonical benchmark for all languages."
)

as: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
bn: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
gu: queries=999, documents=999, BM25=(999, 999), E5=(999, 999)
hi: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
kn: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
ml: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
mr: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
ne: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
or: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
pa: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
ta: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
te: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)
ur: queries=1000, documents=1000, BM25=(1000, 1000), E5=(1000, 1000)

PASS: BM25 and E5 matrices match the canonical benchmark for all languages.


In [35]:
lang = "hi"

queries, corpus, gold = load_benchmark(lang)

bm25_scores = np.load(
    BM25_DIR / f"{lang}_scores.npy"
)

e5_scores = np.load(
    E5_DIR / f"{lang}_scores.npy"
)

expected_shape = (
    len(queries),
    len(corpus),
)

assert bm25_scores.shape == expected_shape
assert e5_scores.shape == expected_shape
assert len(gold) == len(queries)

print("Language:", lang)
print("Queries:", len(queries))
print("Documents:", len(corpus))
print("BM25 shape:", bm25_scores.shape)
print("E5 shape:", e5_scores.shape)

Language: hi
Queries: 1000
Documents: 1000
BM25 shape: (1000, 1000)
E5 shape: (1000, 1000)


In [36]:
t0 = time.time()

hybrid_scores = rrf_fuse(
    score_mats=[
        bm25_scores,
        e5_scores,
    ],
    k=RRF_K,
    depth=RRF_DEPTH,
)

elapsed = time.time() - t0

assert hybrid_scores.shape == expected_shape
assert np.isfinite(hybrid_scores).all()

print("Hybrid matrix shape:", hybrid_scores.shape)
print("Fusion time:", f"{elapsed:.3f} seconds")

Hybrid matrix shape: (1000, 1000)
Fusion time: 0.321 seconds


In [37]:
metrics = record(
    model=HYBRID_TAG,
    key=lang,
    S=hybrid_scores,
    gold=gold,
    setting="mono",
)

print("\n" + "=" * 65)
print(f"BM25 + multilingual-E5 RRF: {lang}")
print("=" * 65)

print(f"MRR:        {metrics['MRR']:.4f}")
print(f"MRR@10:     {metrics['MRR@10']:.4f}")
print(f"Recall@1:   {metrics['R@1']:.4f}")
print(f"Recall@5:   {metrics['R@5']:.4f}")
print(f"Recall@10:  {metrics['R@10']:.4f}")
print(f"Recall@20:  {metrics['R@20']:.4f}")
print(f"Recall@100: {metrics['R@100']:.4f}")


BM25 + multilingual-E5 RRF: hi
MRR:        0.8019
MRR@10:     0.7984
Recall@1:   0.7370
Recall@5:   0.8730
Recall@10:  0.9060
Recall@20:  0.9360
Recall@100: 0.9840


In [38]:
all_metrics = []
failed_languages = []

for lang in LANGS:

    print("\n" + "=" * 65)
    print(
        f"Running BM25 + multilingual-E5 RRF: {lang}"
    )
    print("=" * 65)

    try:

        start = time.time()

        # ----------------------------------------------------
        # Load saved baseline score matrices
        # ----------------------------------------------------

        bm25_scores = np.load(
            BM25_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        e5_scores = np.load(
            E5_DIR / f"{lang}_scores.npy",
            mmap_mode="r",
        )

        # ----------------------------------------------------
        # Load canonical benchmark
        # ----------------------------------------------------

        queries, corpus, gold = load_benchmark(
            lang
        )

        expected_shape = (
            len(queries),
            len(corpus),
        )

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        assert bm25_scores.shape == expected_shape, (
            f"BM25 shape mismatch: "
            f"{bm25_scores.shape} != {expected_shape}"
        )

        assert e5_scores.shape == expected_shape, (
            f"E5 shape mismatch: "
            f"{e5_scores.shape} != {expected_shape}"
        )

        assert len(gold) == len(queries)

        # ----------------------------------------------------
        # RRF
        # ----------------------------------------------------

        hybrid_scores = rrf_fuse(
            score_mats=[
                bm25_scores,
                e5_scores,
            ],
            k=RRF_K,
            depth=RRF_DEPTH,
        )

        assert hybrid_scores.shape == expected_shape
        assert np.isfinite(hybrid_scores).all()

        # ----------------------------------------------------
        # Save + evaluate
        # ----------------------------------------------------

        metrics = record(
            model=HYBRID_TAG,
            key=lang,
            S=hybrid_scores,
            gold=gold,
            setting="mono",
        )

        all_metrics.append(metrics)

        elapsed = time.time() - start

        print(
            f"MRR:        {metrics['MRR']:.4f}"
        )

        print(
            f"MRR@10:     {metrics['MRR@10']:.4f}"
        )

        print(
            f"Recall@1:   {metrics['R@1']:.4f}"
        )

        print(
            f"Recall@5:   {metrics['R@5']:.4f}"
        )

        print(
            f"Recall@10:  {metrics['R@10']:.4f}"
        )

        print(
            f"Recall@20:  {metrics['R@20']:.4f}"
        )

        print(
            f"Recall@100: {metrics['R@100']:.4f}"
        )

        print(
            f"Time: {elapsed:.2f} seconds"
        )

        del (
            bm25_scores,
            e5_scores,
            hybrid_scores,
        )

        gc.collect()

    except Exception as exc:

        print(
            f"FAILED: {lang}"
        )

        print(
            f"Error: {exc}"
        )

        failed_languages.append(lang)

        gc.collect()


print("\n" + "=" * 65)
print("BM25 + multilingual-E5 RRF RUN FINISHED")
print("=" * 65)

print(
    "Successful this session:",
    len(all_metrics),
)

print(
    "Failed languages:",
    failed_languages or "None",
)


Running BM25 + multilingual-E5 RRF: as
MRR:        0.5897
MRR@10:     0.5791
Recall@1:   0.5080
Recall@5:   0.6730
Recall@10:  0.7250
Recall@20:  0.8180
Recall@100: 0.9570
Time: 0.32 seconds

Running BM25 + multilingual-E5 RRF: bn
MRR:        0.7321
MRR@10:     0.7261
Recall@1:   0.6590
Recall@5:   0.8120
Recall@10:  0.8500
Recall@20:  0.9010
Recall@100: 0.9780
Time: 0.27 seconds

Running BM25 + multilingual-E5 RRF: gu
MRR:        0.6499
MRR@10:     0.6423
Recall@1:   0.5656
Recall@5:   0.7407
Recall@10:  0.7898
Recall@20:  0.8438
Recall@100: 0.9610
Time: 0.33 seconds

Running BM25 + multilingual-E5 RRF: hi
MRR:        0.8019
MRR@10:     0.7984
Recall@1:   0.7370
Recall@5:   0.8730
Recall@10:  0.9060
Recall@20:  0.9360
Recall@100: 0.9840
Time: 0.41 seconds

Running BM25 + multilingual-E5 RRF: kn
MRR:        0.6456
MRR@10:     0.6348
Recall@1:   0.5770
Recall@5:   0.7030
Recall@10:  0.7560
Recall@20:  0.8690
Recall@100: 0.9690
Time: 0.66 seconds

Running BM25 + multilingual-E5 RRF: ml


In [39]:
verification_errors = []

for lang in LANGS:

    scores_path = (
        HYBRID_DIR / f"{lang}_scores.npy"
    )

    ranks_path = (
        HYBRID_DIR / f"{lang}_ranks.npy"
    )

    if not scores_path.exists():

        verification_errors.append(
            f"{lang}: missing scores file"
        )

        continue

    if not ranks_path.exists():

        verification_errors.append(
            f"{lang}: missing ranks file"
        )

        continue

    queries, corpus, gold = load_benchmark(
        lang
    )

    scores = np.load(
        scores_path,
        mmap_mode="r",
    )

    ranks = np.load(
        ranks_path
    )

    expected_scores_shape = (
        len(queries),
        len(corpus),
    )

    expected_ranks_shape = (
        len(queries),
    )

    if scores.shape != expected_scores_shape:

        verification_errors.append(
            f"{lang}: incorrect score shape "
            f"{scores.shape}"
        )

    if ranks.shape != expected_ranks_shape:

        verification_errors.append(
            f"{lang}: incorrect rank shape "
            f"{ranks.shape}"
        )

    if not np.isfinite(scores).all():

        verification_errors.append(
            f"{lang}: non-finite scores"
        )

    if np.any(ranks < 1) or np.any(
        ranks > len(corpus)
    ):

        verification_errors.append(
            f"{lang}: invalid gold rank"
        )

    print(
        f"{lang}: "
        f"scores={scores.shape}, "
        f"ranks={ranks.shape}, "
        f"verified"
    )

print("\nScore files:",
      sum(
          (HYBRID_DIR / f"{l}_scores.npy").exists()
          for l in LANGS
      ),
      "/ 13")

print(
    "Rank files:",
    sum(
        (HYBRID_DIR / f"{l}_ranks.npy").exists()
        for l in LANGS
    ),
    "/ 13",
)

if verification_errors:

    print("\nVERIFICATION FAILED:")

    for error in verification_errors:
        print("-", error)

    raise RuntimeError(
        "Output verification failed."
    )

print(
    "\nPASS: all BM25 + E5 hybrid outputs verified."
)

as: scores=(1000, 1000), ranks=(1000,), verified
bn: scores=(1000, 1000), ranks=(1000,), verified
gu: scores=(999, 999), ranks=(999,), verified
hi: scores=(1000, 1000), ranks=(1000,), verified
kn: scores=(1000, 1000), ranks=(1000,), verified
ml: scores=(1000, 1000), ranks=(1000,), verified
mr: scores=(1000, 1000), ranks=(1000,), verified
ne: scores=(1000, 1000), ranks=(1000,), verified
or: scores=(1000, 1000), ranks=(1000,), verified
pa: scores=(1000, 1000), ranks=(1000,), verified
ta: scores=(1000, 1000), ranks=(1000,), verified
te: scores=(1000, 1000), ranks=(1000,), verified
ur: scores=(1000, 1000), ranks=(1000,), verified

Score files: 13 / 13
Rank files: 13 / 13

PASS: all BM25 + E5 hybrid outputs verified.


In [40]:
metrics_path = RES_DIR / "metrics.csv"

assert metrics_path.exists(), (
    "metrics.csv not found."
)

metrics_df = pd.read_csv(
    metrics_path
)

required_models = [
    BM25_TAG,
    E5_TAG,
    HYBRID_TAG,
]

comparison = metrics_df[
    (
        metrics_df["model"]
        .isin(required_models)
    )
    &
    (
        metrics_df["setting"]
        == "mono"
    )
].copy()

comparison = comparison[
    comparison["key"].isin(LANGS)
]

# Keep latest result if a notebook was rerun.
comparison = comparison.drop_duplicates(
    subset=[
        "model",
        "key",
        "setting",
    ],
    keep="last",
)

for model_name in required_models:

    present = set(
        comparison.loc[
            comparison["model"] == model_name,
            "key",
        ]
    )

    missing = (
        set(LANGS)
        - present
    )

    print(
        f"{model_name}: "
        f"{len(present)}/13 languages"
    )

    if missing:
        print(
            "  Missing:",
            sorted(missing),
        )

bm25: 13/13 languages
mE5-base: 13/13 languages
bm25+mE5-base-rrf: 13/13 languages


In [41]:
mrr_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="MRR",
).reindex(LANGS)

print("\nMRR comparison:")

display(
    mrr_comparison.round(4)
)


MRR comparison:


model,bm25,bm25+mE5-base-rrf,mE5-base
key,,,
as,0.4616,0.5897,0.7028
bn,0.6189,0.7321,0.8089
gu,0.5465,0.6499,0.7284
hi,0.7117,0.8019,0.8764
kn,0.5380,0.6456,0.7717
ml,0.5248,0.6411,0.7747
mr,0.5097,0.6577,0.8001
ne,0.5052,0.6399,0.7900
or,0.5172,0.6244,0.7085


In [42]:
r10_comparison = comparison.pivot(
    index="key",
    columns="model",
    values="R@10",
).reindex(LANGS)

print("\nRecall@10 comparison:")

display(
    r10_comparison.round(4)
)


Recall@10 comparison:


model,bm25,bm25+mE5-base-rrf,mE5-base
key,,,
as,0.5990,0.7250,0.8510
bn,0.7720,0.8500,0.9200
gu,0.6877,0.7898,0.8589
hi,0.8420,0.9060,0.9530
kn,0.6630,0.7560,0.8840
ml,0.6620,0.7700,0.8920
mr,0.6630,0.7690,0.9130
ne,0.6610,0.7950,0.9030
or,0.6520,0.7610,0.8400


In [43]:
comparison_wide = mrr_comparison.copy()

comparison_wide["Hybrid_minus_BM25"] = (
    comparison_wide[HYBRID_TAG]
    - comparison_wide[BM25_TAG]
)

comparison_wide["Hybrid_minus_E5"] = (
    comparison_wide[HYBRID_TAG]
    - comparison_wide[E5_TAG]
)

comparison_wide["Best_baseline_MRR"] = (
    comparison_wide[
        [BM25_TAG, E5_TAG]
    ].max(axis=1)
)

comparison_wide[
    "Hybrid_beats_best_baseline"
] = (
    comparison_wide[HYBRID_TAG]
    >
    comparison_wide["Best_baseline_MRR"]
)

display(
    comparison_wide.round(4)
)

model,bm25,bm25+mE5-base-rrf,mE5-base,Hybrid_minus_BM25,Hybrid_minus_E5,Best_baseline_MRR,Hybrid_beats_best_baseline
key,,,,,,,
as,0.4616,0.5897,0.7028,0.1281,-0.1131,0.7028,False
bn,0.6189,0.7321,0.8089,0.1133,-0.0768,0.8089,False
gu,0.5465,0.6499,0.7284,0.1033,-0.0785,0.7284,False
hi,0.7117,0.8019,0.8764,0.0902,-0.0745,0.8764,False
kn,0.5380,0.6456,0.7717,0.1075,-0.1261,0.7717,False
ml,0.5248,0.6411,0.7747,0.1163,-0.1336,0.7747,False
mr,0.5097,0.6577,0.8001,0.1480,-0.1424,0.8001,False
ne,0.5052,0.6399,0.7900,0.1347,-0.1501,0.7900,False
or,0.5172,0.6244,0.7085,0.1073,-0.0840,0.7085,False


In [44]:
print("\n" + "=" * 75)
print("BM25 + multilingual-E5 RRF SUMMARY")
print("=" * 75)

print(
    "Hybrid tag:",
    HYBRID_TAG
)

print(
    "RRF k:",
    RRF_K
)

print(
    "RRF depth:",
    RRF_DEPTH
)

print(
    "Languages evaluated:",
    len(
        comparison_wide.dropna(
            subset=[HYBRID_TAG]
        )
    ),
)

print(
    "Languages where hybrid beats BM25:",
    int(
        (
            comparison_wide["Hybrid_minus_BM25"]
            > 0
        ).sum()
    ),
)

print(
    "Languages where hybrid beats E5:",
    int(
        (
            comparison_wide["Hybrid_minus_E5"]
            > 0
        ).sum()
    ),
)

print(
    "Languages where hybrid beats BOTH:",
    int(
        comparison_wide[
            "Hybrid_beats_best_baseline"
        ].sum()
    ),
)

print(
    "\nMean MRR:"
)

print(
    f"BM25:  {mrr_comparison[BM25_TAG].mean():.4f}"
)

print(
    f"E5:    {mrr_comparison[E5_TAG].mean():.4f}"
)

print(
    f"Hybrid: {mrr_comparison[HYBRID_TAG].mean():.4f}"
)


BM25 + multilingual-E5 RRF SUMMARY
Hybrid tag: bm25+mE5-base-rrf
RRF k: 60
RRF depth: None
Languages evaluated: 13
Languages where hybrid beats BM25: 13
Languages where hybrid beats E5: 0
Languages where hybrid beats BOTH: 0

Mean MRR:
BM25:  0.5651
E5:    0.7733
Hybrid: 0.6790


In [45]:
# BM25 + multilingual-E5 RRF experiments

BM25_TAG = "bm25"

E5_VARIANTS = [
    "mE5-small",
    "mE5-base",
    "mE5-large",
]

RRF_K = 60
RRF_DEPTH = None

print("E5 variants:", E5_VARIANTS)
print("RRF k:", RRF_K)
print("RRF depth:", RRF_DEPTH)

E5 variants: ['mE5-small', 'mE5-base', 'mE5-large']
RRF k: 60
RRF depth: None


In [46]:
for e5_tag in E5_VARIANTS:
    e5_dir = RUN_DIR / e5_tag

    assert e5_dir.is_dir(), (
        f"Missing E5 directory: {e5_dir}"
    )

    for lang in LANGS:
        for suffix in ["scores", "ranks"]:
            path = e5_dir / f"{lang}_{suffix}.npy"

            assert path.exists(), (
                f"Missing file: {path}"
            )

print("PASS: all E5 variants have scores and ranks for all languages.")

PASS: all E5 variants have scores and ranks for all languages.


In [47]:
import gc
import time
import numpy as np

all_metrics = []
failed_experiments = []

for e5_tag in E5_VARIANTS:

    hybrid_tag = f"bm25+{e5_tag}-rrf"

    print("\n" + "=" * 75)
    print(f"EXPERIMENT: {hybrid_tag}")
    print("=" * 75)

    e5_dir = RUN_DIR / e5_tag

    for lang in LANGS:

        start = time.time()

        try:
            queries, corpus, gold = load_benchmark(lang)

            bm25_scores = np.load(
                RUN_DIR / BM25_TAG / f"{lang}_scores.npy",
                mmap_mode="r",
            )

            e5_scores = np.load(
                e5_dir / f"{lang}_scores.npy",
                mmap_mode="r",
            )

            expected_shape = (
                len(queries),
                len(corpus),
            )

            assert bm25_scores.shape == expected_shape, (
                f"BM25 shape mismatch for {lang}"
            )

            assert e5_scores.shape == expected_shape, (
                f"{e5_tag} shape mismatch for {lang}"
            )

            assert len(gold) == len(queries)

            hybrid_scores = rrf_fuse(
                score_mats=[
                    bm25_scores,
                    e5_scores,
                ],
                k=RRF_K,
                depth=RRF_DEPTH,
            )

            assert hybrid_scores.shape == expected_shape
            assert np.isfinite(hybrid_scores).all()

            metrics = record(
                model=hybrid_tag,
                key=lang,
                S=hybrid_scores,
                gold=gold,
                setting="mono",
            )

            all_metrics.append(metrics)

            print(
                f"{e5_tag:10s} | {lang:2s} | "
                f"MRR={metrics['MRR']:.4f} | "
                f"MRR@10={metrics['MRR@10']:.4f} | "
                f"R@10={metrics['R@10']:.4f} | "
                f"{time.time() - start:.2f}s"
            )

            del (
                bm25_scores,
                e5_scores,
                hybrid_scores,
            )

        except Exception as exc:
            failed_experiments.append({
                "model": hybrid_tag,
                "language": lang,
                "error": str(exc),
            })

            print(
                f"FAILED | {hybrid_tag} | "
                f"{lang} | {exc}"
            )

        finally:
            gc.collect()

print("\nAll experiments finished.")
print("Successful:", len(all_metrics), "/ 39")
print("Failures:", len(failed_experiments))

if failed_experiments:
    display(pd.DataFrame(failed_experiments))


EXPERIMENT: bm25+mE5-small-rrf
mE5-small  | as | MRR=0.5752 | MRR@10=0.5637 | R@10=0.7110 | 0.25s
mE5-small  | bn | MRR=0.7151 | MRR@10=0.7084 | R@10=0.8370 | 0.30s
mE5-small  | gu | MRR=0.6506 | MRR@10=0.6424 | R@10=0.7798 | 0.38s
mE5-small  | hi | MRR=0.8028 | MRR@10=0.7992 | R@10=0.9060 | 0.41s
mE5-small  | kn | MRR=0.6454 | MRR@10=0.6347 | R@10=0.7500 | 0.26s
mE5-small  | ml | MRR=0.6460 | MRR@10=0.6357 | R@10=0.7660 | 0.32s
mE5-small  | mr | MRR=0.6522 | MRR@10=0.6428 | R@10=0.7690 | 0.82s
mE5-small  | ne | MRR=0.6393 | MRR@10=0.6306 | R@10=0.8010 | 0.73s
mE5-small  | or | MRR=0.6256 | MRR@10=0.6173 | R@10=0.7660 | 0.89s
mE5-small  | pa | MRR=0.7188 | MRR@10=0.7133 | R@10=0.8490 | 0.93s
mE5-small  | ta | MRR=0.6498 | MRR@10=0.6412 | R@10=0.7800 | 0.30s
mE5-small  | te | MRR=0.7241 | MRR@10=0.7178 | R@10=0.8280 | 0.45s
mE5-small  | ur | MRR=0.7289 | MRR@10=0.7236 | R@10=0.8620 | 0.41s

EXPERIMENT: bm25+mE5-base-rrf
mE5-base   | as | MRR=0.5897 | MRR@10=0.5791 | R@10=0.7250 | 0.29s

,model,language,error
0,bm25+mE5-base-rrf,ur,[Errno 22] Invalid argument: 'C:\\Users\\sanka...


In [48]:
metrics_df = pd.read_csv(
    RES_DIR / "metrics.csv"
)

hybrid_tags = [
    f"bm25+{tag}-rrf"
    for tag in E5_VARIANTS
]

models_to_compare = [
    "bm25",
    "mE5-small",
    "mE5-base",
    "mE5-large",
    "bm25+bge-m3-rrf",
    *hybrid_tags,
]

comparison = metrics_df[
    metrics_df["model"].isin(models_to_compare)
    & (metrics_df["setting"] == "mono")
    & metrics_df["key"].isin(LANGS)
].copy()

comparison = comparison.drop_duplicates(
    subset=["model", "key", "setting"],
    keep="last",
)

mrr_table = comparison.pivot(
    index="key",
    columns="model",
    values="MRR",
).reindex(LANGS)

display(mrr_table.round(4))

model,bm25,bm25+bge-m3-rrf,bm25+mE5-base-rrf,bm25+mE5-large-rrf,bm25+mE5-small-rrf,mE5-base,mE5-large,mE5-small
key,,,,,,,,
as,0.4616,0.6094,0.5897,0.6075,0.5752,0.7028,0.7706,0.6324
bn,0.6189,0.7397,0.7321,0.7454,0.7151,0.8089,0.8709,0.7773
gu,0.5465,0.6752,0.6499,0.6687,0.6506,0.7284,0.8035,0.6900
hi,0.7117,0.8005,0.8019,0.8094,0.8028,0.8764,0.9101,0.8788
kn,0.5380,0.6542,0.6456,0.6506,0.6454,0.7717,0.8193,0.7616
ml,0.5248,0.6515,0.6411,0.6624,0.6460,0.7747,0.8406,0.7548
mr,0.5097,0.6683,0.6577,0.6680,0.6522,0.8001,0.8539,0.7726
ne,0.5052,0.6557,0.6399,0.6538,0.6393,0.7900,0.8446,0.7858
or,0.5172,0.6477,0.6244,0.6434,0.6256,0.7085,0.7815,0.6827


In [49]:
mean_mrr = mrr_table.mean(axis=0).sort_values(
    ascending=False
)

print("Mean MRR across languages:")
display(mean_mrr.to_frame("Mean MRR").round(4))

Mean MRR across languages:


,Mean MRR
model,
mE5-large,0.8343
mE5-base,0.7733
mE5-small,0.7469
bm25+mE5-large-rrf,0.6942
bm25+bge-m3-rrf,0.6930
bm25+mE5-base-rrf,0.6790
bm25+mE5-small-rrf,0.6749
bm25,0.5651


In [50]:
import pandas as pd
from pathlib import Path

metrics_path = RES_DIR / "metrics.csv"
metrics_df = pd.read_csv(metrics_path)

# Inspect available model names to verify exact tags
print("Available models:")
print(sorted(metrics_df["model"].unique()))

# Include standalone models and their RRF hybrids
models_to_compare = [
    "mE5-large",
    "mE5-base",
    "mE5-small",
    "bge-m3",             # Standalone BGE-M3
    "bm25+mE5-large-rrf",
    "bm25+bge-m3-rrf",
    "bm25+mE5-base-rrf",
    "bm25+mE5-small-rrf",
    "bm25",
]

# Keep monolingual results across the benchmark languages
df = metrics_df[
    (metrics_df["setting"] == "mono")
    & (metrics_df["key"].isin(LANGS))
    & (metrics_df["model"].isin(models_to_compare))
].copy()

# Calculate macro-average MRR across languages
summary = (
    df.groupby("model")["MRR"]
    .mean()
    .reindex(models_to_compare)
    .dropna()
    .rename("Mean MRR")
    .reset_index()
)

summary = summary.sort_values("Mean MRR", ascending=False).reset_index(drop=True)
display(summary.round(4))

Available models:
['bge-m3', 'bm25', 'bm25+bge-m3-rrf', 'bm25+mE5-base-rrf', 'bm25+mE5-large-rrf', 'bm25+mE5-small-rrf', 'indicbert-v3', 'mE5-base', 'mE5-large', 'mE5-small', 'muril']


,model,Mean MRR
0,bge-m3,0.8384
1,mE5-large,0.8343
2,mE5-base,0.7733
3,mE5-small,0.7469
4,bm25+mE5-large-rrf,0.6942
5,bm25+bge-m3-rrf,0.6930
6,bm25+mE5-base-rrf,0.6790
7,bm25+mE5-small-rrf,0.6749
8,bm25,0.5651
